In [29]:
import pyspiel
import numpy as np

game = pyspiel.load_game("connect_four")

In [30]:
class MCTSNode_TT:
    def __init__(self, state: pyspiel.State, parent_action = None, parent = None):
        self.state = state
        self.parent_action = parent_action
        self.parent = parent
        self.children = {}
        self.visits = 0
        self.value = 0.0
        self.untried_actions = state.legal_actions() if state is not None else []
    
    def is_fully_expanded(self):
        return len(self.untried_actions) == 0
    
    def best_child(self, exploration_weight: float = 1.4):
        best_score = float('-inf')
        best_child = None
        for action, child in self.children.items():
            exploit = child.value / child.visits
            explore = np.sqrt(np.log(self.visits) / child.visits)
            score = exploit + exploration_weight * explore
            if score > best_score:
                best_score = score
                best_child = child
        return best_child

In [51]:
class MCTS_TT:
    def __init__(self, game: pyspiel.Game, exploration_weight: float = 1.4, iterations: int = 1000):
        self.game = game
        self.iterations = iterations
        self.exploration_weight = exploration_weight

        self.transposition_table = {}
     
    def get_node(self, state: pyspiel.State, parent = None, parent_action = None):
        state_str = str(state)
        if state_str not in self.transposition_table:
            self.transposition_table[state_str] = MCTSNode_TT(state.clone(), parent_action, parent)
        return self.transposition_table[state_str]
    
    def search(self, state: pyspiel.State):
        root = self.get_node(state)
        for _ in range(self.iterations):
            node = root
            # Selection
            while node.is_fully_expanded() and not node.state.is_terminal():
                node = node.best_child(self.exploration_weight)
            # Expansion
            if not node.state.is_terminal():
                action = node.untried_actions.pop()
                next_state = node.state.clone()
                next_state.apply_action(action)
                # check transposition table
                child_node = self.get_node(next_state, node, action)
                node.children[action] = child_node
                node = child_node
            # Simulation
            current_state = node.state.clone()
            while not current_state.is_terminal():
                legal_actions = current_state.legal_actions()
                current_state.apply_action(np.random.choice(legal_actions))
            # Backpropagation
            reward = current_state.returns()[root.state.current_player()]
            temp_node = node
            while temp_node is not None:
                temp_node.visits += 1
                temp_node.value += reward
                temp_node = temp_node.parent
                # break condition to avoid infinite loop in case of root's parent
                if temp_node is None or temp_node == root.parent:
                    break
        # select best move based on most visits
        return max(root.children.items(), key=lambda item: item[1].visits)[0]

In [55]:
mcts_tt = MCTS_TT(game, exploration_weight=1.4, iterations=2000)
num_games = 100
win, lose, draw = 0, 0, 0
print(f'Starting MCTS with Transposition Table on Connect Four for {num_games} games.')
for i in range(num_games):
    state = game.new_initial_state()
    while not state.is_terminal():
        if state.current_player() == 0:
            # print(f'Player 0 (MCTS TT) to move. Current state:\n{state}')
            action = mcts_tt.search(state)
        else:
            legal_actions = state.legal_actions()
            action = np.random.choice(legal_actions)
        state.apply_action(action)
    print(f'Game {i+1} finished. Returns: {state.returns()}')
    if state.returns()[0] > state.returns()[1]:
        win += 1
    elif state.returns()[0] < state.returns()[1]:
        lose += 1
    else:
        draw += 1
print(f'Out of {num_games} games: Wins: {win}, Losses: {lose}, Draws: {draw}')

Starting MCTS with Transposition Table on Connect Four for 100 games.
Game 1 finished. Returns: [1.0, -1.0]
Game 2 finished. Returns: [1.0, -1.0]
Game 3 finished. Returns: [1.0, -1.0]
Game 4 finished. Returns: [1.0, -1.0]
Game 5 finished. Returns: [1.0, -1.0]
Game 6 finished. Returns: [1.0, -1.0]
Game 7 finished. Returns: [1.0, -1.0]
Game 8 finished. Returns: [1.0, -1.0]
Game 9 finished. Returns: [1.0, -1.0]
Game 10 finished. Returns: [1.0, -1.0]
Game 11 finished. Returns: [1.0, -1.0]
Game 12 finished. Returns: [1.0, -1.0]
Game 13 finished. Returns: [1.0, -1.0]
Game 14 finished. Returns: [1.0, -1.0]
Game 15 finished. Returns: [1.0, -1.0]
Game 16 finished. Returns: [1.0, -1.0]
Game 17 finished. Returns: [1.0, -1.0]
Game 18 finished. Returns: [1.0, -1.0]
Game 19 finished. Returns: [1.0, -1.0]
Game 20 finished. Returns: [1.0, -1.0]
Game 21 finished. Returns: [1.0, -1.0]
Game 22 finished. Returns: [1.0, -1.0]
Game 23 finished. Returns: [1.0, -1.0]
Game 24 finished. Returns: [1.0, -1.0]
Gam

In [58]:
# human vs mcts with tt
state = game.new_initial_state()
while not state.is_terminal():
    if state.current_player() == 1:
        print(f'Player 0 (MCTS TT) to move. Current state:\n{state}')
        action = mcts_tt.search(state)
    else:
        print(f'Player 1 (Human) to move. Current state:\n{state}')
        legal_actions = state.legal_actions()
        print(f'Legal actions: {legal_actions}')
        action = int(input('Enter your action: '))
        while action not in legal_actions:
            print('Invalid action. Try again.')
            action = int(input('Enter your action: '))
    state.apply_action(action)
print(f'Game finished. Returns: {state.returns()}')

Player 1 (Human) to move. Current state:
.......
.......
.......
.......
.......
.......

Legal actions: [0, 1, 2, 3, 4, 5, 6]
Player 0 (MCTS TT) to move. Current state:
.......
.......
.......
.......
.......
...x...

Player 1 (Human) to move. Current state:
.......
.......
.......
.......
.......
..ox...

Legal actions: [0, 1, 2, 3, 4, 5, 6]
Player 0 (MCTS TT) to move. Current state:
.......
.......
.......
.......
..x....
..ox...

Player 1 (Human) to move. Current state:
.......
.......
.......
.......
..x....
..ox.o.

Legal actions: [0, 1, 2, 3, 4, 5, 6]
Player 0 (MCTS TT) to move. Current state:
.......
.......
.......
.......
..xx...
..ox.o.

Player 1 (Human) to move. Current state:
.......
.......
.......
...o...
..xx...
..ox.o.

Legal actions: [0, 1, 2, 3, 4, 5, 6]
Player 0 (MCTS TT) to move. Current state:
.......
.......
.......
...o...
..xx...
..oxxo.

Player 1 (Human) to move. Current state:
.......
.......
.......
...o...
..xxo..
..oxxo.

Legal actions: [0, 1, 2, 3, 4, 5, 